In [2]:
import sys
print(sys.executable)

c:\Users\mainu\Desktop\Facial Emotion Recognition\ann\Scripts\python.exe


In [1]:
# ==========================================
# Import Libraries
# ==========================================

import os
import cv2
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# ==========================================
# Check Library Versions
# ==========================================

import tensorflow as tf

print("TensorFlow :", tf.__version__)
print("MediaPipe  :", mp.__version__)
print("OpenCV     :", cv2.__version__)
print("NumPy      :", np.__version__)
print("Pandas     :", pd.__version__)

TensorFlow : 2.16.2
MediaPipe  : 0.10.14
OpenCV     : 4.10.0
NumPy      : 1.26.4
Pandas     : 2.2.3


In [37]:
# ==========================================
# Load Face Landmarker Model
# ==========================================

MODEL_PATH = "../models/face_landmarker.task"

base_options = python.BaseOptions(
    model_asset_path=MODEL_PATH
)

options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False,
    num_faces=1
)

detector = vision.FaceLandmarker.create_from_options(options)

print("Face Landmarker Loaded Successfully!")

Face Landmarker Loaded Successfully!


In [3]:
from pathlib import Path

print("Current Working Directory:")
print(Path.cwd())

Current Working Directory:
c:\Users\mainu\Desktop\Facial Emotion Recognition\notebooks


In [4]:
from pathlib import Path

for item in Path("../").iterdir():
    print(item)

..\ann
..\app.py
..\dataset
..\models
..\notebooks
..\requirements.txt
..\utils


In [5]:
from pathlib import Path

DATASET_PATH = Path("../dataset")

print("Contents of dataset folder:\n")

for item in DATASET_PATH.iterdir():
    print(item)

Contents of dataset folder:

..\dataset\emotion.csv
..\dataset\landmark_dataset_v2.csv


In [6]:
# ==========================================
# Verify Dataset Path
# ==========================================

from pathlib import Path

DATASET_PATH = Path("../dataset/emotion.csv/train")

print("Exists:", DATASET_PATH.exists())

emotion_classes = sorted([
    folder.name
    for folder in DATASET_PATH.iterdir()
    if folder.is_dir()
])

print("\nClasses:")
print(emotion_classes)

print("\nTotal Classes:", len(emotion_classes))

Exists: True

Classes:
['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

Total Classes: 7


In [7]:
# ==========================================
# Count Images in Each Emotion
# ==========================================

emotion_counts = {}

for emotion in emotion_classes:

    emotion_folder = DATASET_PATH / emotion

    count = len(list(emotion_folder.glob("*")))

    emotion_counts[emotion] = count

emotion_df = pd.DataFrame(
    emotion_counts.items(),
    columns=["Emotion", "Images"]
)

emotion_df

,Emotion,Images
0,angry,3995
1,disgust,436
2,fear,4097
3,happy,7215
4,neutral,4965
5,sad,4830
6,surprise,3171


In [8]:
# ==========================================
# Improved Landmark Extraction Function
# ==========================================

def detect_face(image_rgb):

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=image_rgb
    )

    result = detector.detect(mp_image)

    if not result.face_landmarks:
        return None

    landmarks = result.face_landmarks[0]

    if len(landmarks) != 478:
        return None

    # ==========================================
    # Normalize Landmarks
    # ==========================================

    coords = np.array([[lm.x, lm.y, lm.z] for lm in landmarks],dtype=np.float32)

    # Face center
    center = coords.mean(axis=0)

    # Translate landmarks
    coords = coords - center

    # Scale landmarks
    scale = np.max(np.linalg.norm(coords, axis=1))

    if scale > 0:
        coords = coords / scale

    features = coords.flatten().tolist()

    if len(features) != 1434:
        return None

    return features 


def extract_landmarks(image_path):

    image = cv2.imread(str(image_path))

    if image is None:
        return None

    # First Attempt
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    features = detect_face(image_rgb)

    if features is not None:
        return features

    # Second Attempt (CLAHE)

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8,8)
    )

    enhanced = clahe.apply(gray)

    enhanced_rgb = cv2.cvtColor(
        enhanced,
        cv2.COLOR_GRAY2RGB
    )

    features = detect_face(enhanced_rgb)

    if features is not None:
        return features

    return None

In [ ]:
# ==========================================
# Test Landmark Function
# ==========================================

sample_image = next((DATASET_PATH / "happy").glob("*.*"))

features = extract_landmarks(sample_image)

if features is None:
    print("No face detected!")

else:
    print("Feature Length :", len(features))
    print("First 15 Features:")
    print(features[:15])

NameError: name 'detector' is not defined

In [59]:
# ==========================================
# Generate Landmark Dataset
# ==========================================

dataset = []

processed_images = 0
skipped_images = 0

skipped_files = []

for emotion in emotion_classes:

    print(f"\nProcessing: {emotion}")

    emotion_folder = DATASET_PATH / emotion

    image_paths = list(emotion_folder.glob("*"))

    for image_path in image_paths:

        features = extract_landmarks(image_path)

        if features is None:

            skipped_images += 1
            skipped_files.append(str(image_path))

            continue

        dataset.append(features + [emotion])

        processed_images += 1

print("\n====================================")

print("Dataset Generation Completed")

print("====================================")

print("Processed Images :", processed_images)

print("Skipped Images   :", skipped_images)


Processing: angry

Processing: disgust

Processing: fear

Processing: happy

Processing: neutral

Processing: sad

Processing: surprise

Dataset Generation Completed
Processed Images : 25906
Skipped Images   : 2803


In [60]:
# ==========================================
# Create Landmark DataFrame
# ==========================================

columns = [str(i) for i in range(1434)]

columns.append("Emotion")

landmark_df = pd.DataFrame(
    dataset,
    columns=columns
)

print("Dataset Shape :", landmark_df.shape)

landmark_df.head()

Dataset Shape : (25906, 1435)


,0,1,2,3,4,5,6,7,8,9,...,1425,1426,1427,1428,1429,1430,1431,1432,1433,Emotion
0,0.472540,0.840126,-0.014326,0.495075,0.791484,-0.135946,0.484833,0.792783,-0.045023,0.476306,...,0.659125,0.529080,-0.038623,0.633295,0.551324,-0.038653,0.656321,0.574169,-0.038653,angry
1,0.695906,0.738657,-0.048102,0.745003,0.666726,-0.129881,0.714176,0.681112,-0.059456,0.732245,...,0.856557,0.454991,0.053066,0.827648,0.471215,0.053036,0.844882,0.497969,0.053045,angry
2,0.698148,0.475160,-0.074830,0.711207,0.366087,-0.102533,0.663733,0.423296,-0.063735,0.614943,...,0.550665,0.246345,0.191006,0.546514,0.268771,0.190983,0.563372,0.283947,0.190984,angry
3,0.520289,0.599211,-0.112057,0.491589,0.451643,-0.186951,0.500332,0.502300,-0.105534,0.444835,...,0.642850,0.198585,0.035164,0.609551,0.235866,0.035120,0.650636,0.262663,0.035127,angry
4,0.473564,0.716350,-0.082818,0.446088,0.607351,-0.190320,0.462422,0.639190,-0.089728,0.419138,...,0.693835,0.299642,-0.035064,0.654730,0.335473,-0.035112,0.697249,0.368694,-0.035106,angry


In [61]:
# ==========================================
# Dataset Information
# ==========================================

print(landmark_df.info())

print("\nMissing Values:")

print(landmark_df.isnull().sum().sum())

print("\nEmotion Distribution:")

print(landmark_df["Emotion"].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25906 entries, 0 to 25905
Columns: 1435 entries, 0 to Emotion
dtypes: float64(1434), object(1)
memory usage: 283.6+ MB
None

Missing Values:
0

Emotion Distribution:
Emotion
happy       6861
neutral     4718
sad         4038
fear        3526
angry       3437
surprise    2955
disgust      371
Name: count, dtype: int64


In [63]:
# ==========================================
# Save Skipped Images
# ==========================================

skipped_df = pd.DataFrame(
    skipped_files,
    columns=["Image_Path"]
)

skipped_df.to_csv(
    "../dataset/skipped_images_v2.csv",
    index=False
)

print("Skipped images saved successfully!")

Skipped images saved successfully!


In [64]:
# ==========================================
# Save Landmark Dataset
# ==========================================

output_path = "../dataset/landmark_dataset_v2.csv"

landmark_df.to_csv(
    output_path,
    index=False
)

print("Dataset saved successfully!")

print("Location :", output_path)

Dataset saved successfully!
Location : ../dataset/landmark_dataset_v2.csv


In [65]:
# ==========================================
# Verify Saved Dataset
# ==========================================

verify_df = pd.read_csv(
    "../dataset/landmark_dataset_v2.csv"
)

print("Shape :", verify_df.shape)

verify_df.head()

Shape : (25906, 1435)


,0,1,2,3,4,5,6,7,8,9,...,1425,1426,1427,1428,1429,1430,1431,1432,1433,Emotion
0,0.472540,0.840126,-0.014326,0.495075,0.791484,-0.135946,0.484833,0.792783,-0.045023,0.476306,...,0.659125,0.529080,-0.038623,0.633295,0.551324,-0.038653,0.656321,0.574169,-0.038653,angry
1,0.695906,0.738657,-0.048102,0.745003,0.666726,-0.129881,0.714176,0.681112,-0.059456,0.732245,...,0.856557,0.454991,0.053066,0.827648,0.471215,0.053036,0.844882,0.497969,0.053045,angry
2,0.698148,0.475160,-0.074830,0.711207,0.366087,-0.102533,0.663733,0.423296,-0.063735,0.614943,...,0.550665,0.246345,0.191006,0.546514,0.268771,0.190983,0.563372,0.283947,0.190984,angry
3,0.520289,0.599211,-0.112057,0.491589,0.451643,-0.186951,0.500332,0.502300,-0.105534,0.444835,...,0.642850,0.198585,0.035164,0.609551,0.235866,0.035120,0.650636,0.262663,0.035127,angry
4,0.473564,0.716350,-0.082818,0.446088,0.607351,-0.190320,0.462422,0.639190,-0.089728,0.419138,...,0.693835,0.299642,-0.035064,0.654730,0.335473,-0.035112,0.697249,0.368694,-0.035106,angry
